In [ ]:
# 1/4 - clone main + prototype (fresh, fault-tolerant)
import subprocess, os
print('===== NATIVE LAYERNORM BWD BITEXACT TEST (PURE TKV, NO C INJECTION) =====')
for d,br in [('/content/TVmain','main'),('/content/TVcpu','prototype/cuda-backend')]:
    subprocess.run('rm -rf '+d, shell=True)
    subprocess.run('git clone --depth 1 --branch '+br+' https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
    print(br, subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())

In [ ]:
# 2/4 - build native .so (LN bwd f64 fallback) + .tkv C .so. NO C injection.
# Build CA HAI bang CUNG flags (-O2 -mavx2 -mfma) de FMA contraction khong lech bit.
import subprocess, os
FLAGS='-O2 -mavx2 -mfma'
root='/content/TVmain/Release/tools/simd_native'
srcs='tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c'
p=subprocess.run('cd '+root+' && gcc '+FLAGS+' -shared -fPIC '+srcs+' -lm -lpthread -ldl -o tkv_simd_native.so', shell=True, capture_output=True, text=True, timeout=600)
print('native build rc=',p.returncode)
if p.returncode!=0: print(p.stderr[:2000])

ex='/content/TVcpu/Release/cuda_backend/examples'
cc=ex+'/layernorm_bwd_cpu.cpu.c'; so=ex+'/layernorm_bwd_cpu.so'
txt=open(cc).read()
print('C INJECTED =', 'tkv_lnbwd' in txt, '(expect False = pure TKV)')
print('sqrt(double) present =', 'sqrt(d_var' in txt)
print('gg float present =', 'float f_gg' in txt)
p2=subprocess.run('gcc '+FLAGS+' -shared -fPIC '+cc+' -o '+so+' -lm -lpthread -ldl', shell=True, capture_output=True, text=True, timeout=600)
print('tkv C build rc=',p2.returncode)
if p2.returncode!=0: print(p2.stderr[:2000])

# xac nhan ATen bridge vang -> native.LayernormBwdFused dung f64 fallback
import ctypes, numpy as np
probe=ctypes.CDLL(root+'/tkv_simd_native.so').TkvTorchLayerNormBwd
PP=ctypes.POINTER(ctypes.c_float)
probe.argtypes=[PP,PP,PP,PP,PP,PP,PP,ctypes.c_longlong,ctypes.c_longlong,ctypes.c_double]
probe.restype=ctypes.c_int
z=np.zeros(8,np.float32)
rc=probe(z.ctypes.data_as(PP),z.ctypes.data_as(PP),z.ctypes.data_as(PP),z.ctypes.data_as(PP),
         z.ctypes.data_as(PP),z.ctypes.data_as(PP),z.ctypes.data_as(PP),2,4,1e-6)
print('TkvTorchLayerNormBwd rc=',rc,'(!=0 => native used f64 fallback)')

In [ ]:
# 3/4 - bitwise compare LayernormBwdFused (.tkv pure vs native f64 fallback)
import ctypes, numpy as np
PP=ctypes.POINTER(ctypes.c_float); II=ctypes.c_int; LL=ctypes.c_longlong
args=[PP,II,PP,II,PP,II,PP,II,PP,II,II,II,LL,LL]
native=ctypes.CDLL('/content/TVmain/Release/tools/simd_native/tkv_simd_native.so').LayernormBwdFused
native.argtypes=args; native.restype=II
tkv=ctypes.CDLL('/content/TVcpu/Release/cuda_backend/examples/layernorm_bwd_cpu.so').LayernormBwdFused
tkv.argtypes=args; tkv.restype=II
rng=np.random.default_rng(17)
for n,h in [(4,64),(16,256),(8,768),(7,33)]:
    x=rng.standard_normal(n*h).astype(np.float32)
    gy=rng.standard_normal(n*h).astype(np.float32)
    g=rng.standard_normal(h).astype(np.float32)
    o_dg=0; o_db=h
    # gflat khoi tao KHAC 0 de test duong accumulate dg/db
    gf0=rng.standard_normal(2*h).astype(np.float32)
    ga=gf0.copy(); gc=gf0.copy()
    da=np.zeros(n*h,np.float32); dc=np.zeros(n*h,np.float32)
    native(x.ctypes.data_as(PP),x.size,gy.ctypes.data_as(PP),gy.size,g.ctypes.data_as(PP),g.size,
           da.ctypes.data_as(PP),da.size,ga.ctypes.data_as(PP),ga.size,n,h,o_dg,o_db)
    tkv  (x.ctypes.data_as(PP),x.size,gy.ctypes.data_as(PP),gy.size,g.ctypes.data_as(PP),g.size,
          dc.ctypes.data_as(PP),dc.size,gc.ctypes.data_as(PP),gc.size,n,h,o_dg,o_db)
    ddx=np.count_nonzero(da.view(np.uint32)!=dc.view(np.uint32))
    dgf=np.count_nonzero(ga.view(np.uint32)!=gc.view(np.uint32))
    print('n=%d h=%d dx_bitwise=%s dx_diff=%d/%d gflat_bitwise=%s gflat_diff=%d/%d maxabs=%.3g'%(
        n,h,ddx==0,ddx,da.size,dgf==0,dgf,ga.size,max(np.abs(da-dc).max(),np.abs(ga-gc).max())))

In [ ]:
# 4/4 - summary
print('If all bitwise=True -> LayernormBwdFused pure .tkv reproduces native f64 fallback (NO C injection).')